## 3. EDA Gold — Análise para Camada de Consumo (Silver → Gold)

### a) Imports e path

In [0]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pyspark.sql.functions as F

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(project_root)

### b) Carregando a base Silver

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import Tables

silver_table = f"{Tables.SCHEMA}.{Tables.SILVER_TAXI}"
df_silver = spark.table(silver_table)

total = df_silver.count()
print(f"Tabela: {silver_table}")
print(f"Total de registros: {total:,}  |  Colunas: {len(df_silver.columns)}")
display(df_silver.limit(5))

### c) Completude das colunas exigidas pelo case

O case exige cinco colunas para responder às perguntas de negócio:
- `VendorID` — identificação do fornecedor TPEP
- `tpep_pickup_datetime` — âncora temporal das análises por mês e por hora
- `tpep_dropoff_datetime` — duração da corrida (contexto complementar)
- `passenger_count` — métrica principal de Q2
- `total_amount` — métrica principal de Q1

Para definir os filtros da Gold, vamos medir a completude dessas colunas e quantificar
o impacto de cada valor problemático nas métricas finais.

In [0]:
%load_ext autoreload
%autoreload 2

required_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "total_amount"]

null_exprs = [
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in required_cols
]

df_nulls = df_silver.agg(*null_exprs).toPandas().T
df_nulls.columns = ["nulls"]
df_nulls = df_nulls.reset_index()
df_nulls = df_nulls.rename(columns={"index": "coluna"})
df_nulls["pct_nulo"] = (df_nulls["nulls"] / total * 100).round(4)
df_nulls["registros_validos"] = total - df_nulls["nulls"]
df_nulls["pct_valido"] = (df_nulls["registros_validos"] / total * 100).round(4)

print("Completude das 5 colunas exigidas pelo case:")
display(df_nulls)

# Valores semanticamente problemáticos
null_pass = df_silver.filter(F.col("passenger_count").isNull()).count()
zero_pass = df_silver.filter(F.col("passenger_count") == 0).count()
neg_total = df_silver.filter(F.col("total_amount") <= 0).count()

print(f"\npassenger_count NULL  : {null_pass:,}  ({null_pass/total*100:.2f}%)")
print(f"passenger_count == 0  : {zero_pass:,}  ({zero_pass/total*100:.2f}%)")
print(f"total_amount <= 0     : {neg_total:,}  ({neg_total/total*100:.2f}%)")

# Correlação bidirecional: passenger_count e total_amount são independentes entre si
# Filtrar qualquer um na camada compromete a métrica da outra pergunta
corr_pass_null_amt_ok = df_silver.filter(F.col("passenger_count").isNull() & (F.col("total_amount") > 0)).count()
corr_pass_zero_amt_ok = df_silver.filter((F.col("passenger_count") == 0)   & (F.col("total_amount") > 0)).count()
corr_amt_neg_pass_ok  = df_silver.filter((F.col("total_amount") <= 0)      & (F.col("passenger_count") > 0)).count()

print(f"\nCorrelação bidirecional passenger_count × total_amount:")
print(f"  pass NULL  + amount > 0 : {corr_pass_null_amt_ok:,}  → receita válida para Pergunta 1, inútil para Pergunta 2")
print(f"  pass == 0  + amount > 0 : {corr_pass_zero_amt_ok:,}  → receita válida para Pergunta 1, inútil para Pergunta 2")
print(f"  amount <= 0 + pass > 0  : {corr_amt_neg_pass_ok:,}  → corrida real para Pergunta 2 (chargeback posterior), inútil para Pergunta 1")
print()
print("Conclusão: não existe filtro de camada único correto para ambas as perguntas.")
print("Pergunta 1 e Pergunta 2 têm subconjuntos válidos distintos — os filtros pertencem ao notebook de análise.")

### d) Análise Pergunta 1 — total_amount por mês

**Pergunta do case:** Qual foi o valor total recebido pelos motoristas de táxi amarelo em NYC por mês?

`total_amount` representa o valor final cobrado ao passageiro, incluindo todas as tarifas, impostos e gorjeta.
Valores negativos, confirmados na EDA Silver como chargebacks de `payment_type=4` (Dispute), representam
dinheiro **devolvido** ao passageiro — semanticamente o oposto de "valor recebido".

##### ** Para Pergunta 1, o notebook de análise aplicará `total_amount > 0`. Este filtro não é da camada Gold — a seção c) demonstra que registros com `total_amount <= 0` podem ter `passenger_count > 0` e são necessários para Pergunta 2. **

In [0]:
%load_ext autoreload
%autoreload 2

# Distribuição mensal — Silver completa (sem filtros)
print("Distribuição de total_amount por mês (Silver completa):")
display(
    df_silver
    .groupBy("month")
    .agg(
        F.count("*").alias("corridas"),
        F.round(F.sum("total_amount"), 2).alias("total_amount_sum"),
        F.round(F.avg("total_amount"), 4).alias("total_amount_avg"),
        F.round(F.min("total_amount"), 2).alias("min"),
        F.round(F.max("total_amount"), 2).alias("max"),
    )
    .orderBy("month")
)

# Impacto dos chargebacks na soma mensal
print("\nImpacto dos registros com total_amount <= 0 na soma mensal:")
display(
    df_silver
    .filter(F.col("total_amount") <= 0)
    .groupBy("month")
    .agg(
        F.count("*").alias("registros_negativos_ou_zero"),
        F.round(F.sum("total_amount"), 2).alias("valor_chargeback"),
    )
    .orderBy("month")
)

# Comparativo com vs sem chargebacks
df_q1_com    = df_silver.groupBy("month").agg(F.round(F.sum("total_amount"), 2).alias("total_com_chargeback"))
df_q1_sem    = df_silver.filter(F.col("total_amount") > 0).groupBy("month").agg(F.round(F.sum("total_amount"), 2).alias("total_sem_chargeback"))
df_q1_impact = df_q1_com.join(df_q1_sem, "month").orderBy("month")
df_q1_impact = df_q1_impact.withColumn("diferenca", F.round(F.col("total_sem_chargeback") - F.col("total_com_chargeback"), 2))

print("\nComparativo soma mensal com vs sem chargebacks:")
display(df_q1_impact)

### e) Análise Pergunta 2 — passenger_count por hora em Maio

**Pergunta do case:** Qual foi a quantidade média de passageiros por hora em Maio de 2023?

Dois tipos de registros distorcem a média:
- `passenger_count IS NULL`: falha do fornecedor TPEP (mesmos 428k registros sistêmicos da EDA Silver). O Spark exclui NULLs automaticamente no `avg()`, mas o volume merece documentação.
- `passenger_count == 0`: reposicionamento de veículo ou falha de sensor — não é uma corrida com passageiro. Incluir zeros na média seria semanticamente incorreto para a pergunta proposta.

Nota: registros com `total_amount <= 0` (chargebacks) **não são excluídos** desta análise — a seção c) confirmou que parte deles tem `passenger_count > 0`, ou seja, foram corridas reais com passageiros que tiveram o pagamento disputado posteriormente.

##### ** `passenger_count > 0` e `month == 5` são filtros de query — aplicados no notebook de análise ao responder Pergunta 2, não na camada Gold. **

In [0]:
%load_ext autoreload
%autoreload 2

df_may = df_silver.filter(F.col("month") == 5)
total_may = df_may.count()
print(f"Total de corridas em Maio: {total_may:,}")

# NULLs e zeros em Maio
null_may = df_may.filter(F.col("passenger_count").isNull()).count()
zero_may = df_may.filter(F.col("passenger_count") == 0).count()
valid_may = df_may.filter(F.col("passenger_count") > 0).count()

print(f"\npassenger_count NULL  : {null_may:,}  ({null_may/total_may*100:.4f}%)  → excluído do avg() automaticamente pelo Spark")
print(f"passenger_count == 0  : {zero_may:,}  ({zero_may/total_may*100:.4f}%)  → filtrado na Gold")
print(f"passenger_count > 0   : {valid_may:,}  ({valid_may/total_may*100:.4f}%)  → base para Q2")

# Distribuição por hora (passenger_count > 0, Maio)
print("\nMédia de passageiros por hora em Maio (passenger_count > 0):")
display(
    df_may
    .filter(F.col("passenger_count") > 0)
    .withColumn("hora", F.hour(F.col("tpep_pickup_datetime")))
    .groupBy("hora")
    .agg(
        F.count("*").alias("corridas"),
        F.round(F.avg("passenger_count"), 4).alias("avg_passageiros"),
        F.round(F.sum("passenger_count"), 0).alias("total_passageiros"),
    )
    .orderBy("hora")
)

# Comparativo avg com vs sem zeros
avg_com_zeros = df_may.filter(F.col("passenger_count").isNotNull()).agg(F.avg("passenger_count")).collect()[0][0]
avg_sem_zeros = df_may.filter(F.col("passenger_count") > 0).agg(F.avg("passenger_count")).collect()[0][0]

print(f"\nMédia global em Maio — com zeros    : {avg_com_zeros:.2f} passageiros/corrida")
print(f"Média global em Maio — sem zeros    : {avg_sem_zeros:.2f} passageiros/corrida")
print(f"Diferença (impacto dos zeros)       : {avg_com_zeros - avg_sem_zeros:.2f}")

### f) Distribuição dos campos de código numérico e correlação RatecodeID NULL

`VendorID`, `RatecodeID` e `payment_type` são códigos numéricos cuja semântica está no dicionário da TLC.
A Gold enriquecerá essas colunas com descrições legíveis via `CASE WHEN` para tornar a camada de consumo
auto-explicativa sem depender de joins externos.

Esta seção valida dois pontos:
1. Cobertura do `CASE WHEN` — confirma que todos os valores presentes na Silver estão mapeados.
2. Origem dos NULLs em `RatecodeID` — hipótese: são os mesmos registros do batch sistêmico (`passenger_count IS NULL`), não os registros com `passenger_count == 0`.

In [0]:
%load_ext autoreload
%autoreload 2

# Mapeamentos CASE WHEN que serão aplicados na Gold
vendor_desc_expr = (
    F.when(F.col("VendorID") == 1, "Creative Mobile Technologies (CMT)")
    .when(F.col("VendorID") == 2, "VeriFone Inc.")
    .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("VendorID").cast("string")))
)

ratecode_desc_expr = (
    F.when(F.col("RatecodeID") == 1, "Standard rate")
    .when(F.col("RatecodeID") == 2, "JFK")
    .when(F.col("RatecodeID") == 3, "Newark")
    .when(F.col("RatecodeID") == 4, "Nassau or Westchester")
    .when(F.col("RatecodeID") == 5, "Negotiated fare")
    .when(F.col("RatecodeID") == 6, "Group ride")
    .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("RatecodeID").cast("string")))
)

payment_desc_expr = (
    F.when(F.col("payment_type") == 0, "Flex Fare trip")
    .when(F.col("payment_type") == 1, "Credit card")
    .when(F.col("payment_type") == 2, "Cash")
    .when(F.col("payment_type") == 3, "No charge")
    .when(F.col("payment_type") == 4, "Dispute")
    .when(F.col("payment_type") == 5, "Unknown")
    .when(F.col("payment_type") == 6, "Voided trip")
    .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("payment_type").cast("string")))
)

# VendorID
print("Distribuição VendorID:")
display(
    df_silver
    .withColumn("vendor_desc", vendor_desc_expr)
    .groupBy("VendorID", "vendor_desc")
    .agg(F.count("*").alias("corridas"), F.round(F.count("*") / total * 100, 4).alias("pct"))
    .orderBy("VendorID")
)

# RatecodeID — NULLs aparecem como linha no groupBy
print("Distribuição RatecodeID (NULL visível como linha separada):")
display(
    df_silver
    .withColumn("ratecode_desc", ratecode_desc_expr)
    .groupBy("RatecodeID", "ratecode_desc")
    .agg(F.count("*").alias("corridas"), F.round(F.count("*") / total * 100, 4).alias("pct"))
    .orderBy("RatecodeID")
)

# payment_type
print("Distribuição payment_type:")
display(
    df_silver
    .withColumn("payment_desc", payment_desc_expr)
    .groupBy("payment_type", "payment_desc")
    .agg(F.count("*").alias("corridas"), F.round(F.count("*") / total * 100, 4).alias("pct"))
    .orderBy("payment_type")
)

# Correlação RatecodeID NULL: batch sistêmico ou passenger_count == 0?
# EDA Silver identificou 5 colunas simultaneamente nulas (batch sistêmico):
# passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge, airport_fee
ratecode_null_total   = df_silver.filter(F.col("RatecodeID").isNull()).count()
ratecode_null_pass_null = df_silver.filter(F.col("RatecodeID").isNull() & F.col("passenger_count").isNull()).count()
ratecode_null_pass_zero = df_silver.filter(F.col("RatecodeID").isNull() & (F.col("passenger_count") == 0)).count()

print(f"\nCorrelação RatecodeID NULL:")
print(f"  Total com RatecodeID NULL                        : {ratecode_null_total:,}")
print(f"  RatecodeID NULL + passenger_count NULL (batch)   : {ratecode_null_pass_null:,}  ({ratecode_null_pass_null/ratecode_null_total*100:.2f}%)")
print(f"  RatecodeID NULL + passenger_count == 0           : {ratecode_null_pass_zero:,}  ({ratecode_null_pass_zero/ratecode_null_total*100:.2f}%)")
print()
print("Conclusão: RatecodeID NULL são os mesmos registros do batch sistêmico (passenger_count NULL),")
print("não os registros de passenger_count == 0 — hipóteses independentes, origens distintas.")

### g) Sumário — projeção e enriquecimentos da camada Gold

**Filtros de camada:** nenhum.

A correlação bidirecional documentada na seção c) demonstra que `passenger_count` e `total_amount` são independentes — não existe filtro de camada que seja simultaneamente correto para as Perguntas 1 e 2. \
A Silver já realizou toda a limpeza estrutural; a Gold entrega **projeção de colunas + enriquecimento**. Os filtros de negócio pertencem à pergunta.

**Projeção de colunas:** de 23 colunas da Silver para 9 + 3 enriquecimentos = 12 colunas na Gold.

**Filtros de query** — aplicados no notebook de análise:

| Pergunta | Filtros |
|---|---|
| 1 — valor recebido por mês | `total_amount > 0` |
| 2 — média de passageiros por hora em Maio | `passenger_count > 0`, `month == 5` |

**Enriquecimentos de camada:**

| Coluna original | Coluna gerada | Método |
|---|---|---|
| `VendorID` | `vendor_desc` | `CASE WHEN` |
| `RatecodeID` | `ratecode_desc` | `CASE WHEN` |
| `payment_type` | `payment_desc` | `CASE WHEN` |

**Base Gold:** ~16.1M registros — espelho da Silver com projeção + enriquecimento.

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import Tables

# Projeção: de 23 colunas Silver para as necessárias ao case + year/month de particionamento
GOLD_COLS = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "total_amount",
    "RatecodeID",
    "payment_type",
    "year",
    "month",
]

# Sem filtros de camada — passenger_count e total_amount são independentes (seção c)
df_gold_preview = (
    df_silver
    .select(*GOLD_COLS)
    .withColumn(
        "vendor_desc",
        F.when(F.col("VendorID") == 1, "Creative Mobile Technologies (CMT)")
        .when(F.col("VendorID") == 2, "VeriFone Inc.")
        .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("VendorID").cast("string")))
    )
    .withColumn(
        "ratecode_desc",
        F.when(F.col("RatecodeID") == 1, "Standard rate")
        .when(F.col("RatecodeID") == 2, "JFK")
        .when(F.col("RatecodeID") == 3, "Newark")
        .when(F.col("RatecodeID") == 4, "Nassau or Westchester")
        .when(F.col("RatecodeID") == 5, "Negotiated fare")
        .when(F.col("RatecodeID") == 6, "Group ride")
        .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("RatecodeID").cast("string")))
    )
    .withColumn(
        "payment_desc",
        F.when(F.col("payment_type") == 1, "Credit card")
        .when(F.col("payment_type") == 2, "Cash")
        .when(F.col("payment_type") == 3, "No charge")
        .when(F.col("payment_type") == 4, "Dispute")
        .when(F.col("payment_type") == 5, "Unknown")
        .when(F.col("payment_type") == 6, "Voided trip")
        .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("payment_type").cast("string")))
    )
)

gold_count = df_gold_preview.count()
print(f"Registros Gold: {gold_count:,}  |  Colunas: {len(df_gold_preview.columns)}")
print("\nPré-visualização da camada Gold:")
display(df_gold_preview.limit(10))